# Document parsing

In [1]:
from docx import Document

doc = Document("Timetable-REV7 Hypotheses re press coverage of Hepatitis C in UK +color coding.docx")
doc

In [2]:
from docx.shared import RGBColor
from docx.enum.text import WD_COLOR_INDEX

color_mapping = {
    RGBColor(0xda, 0x73, 0x26): "Non-haemophiliacs with HIV",
    RGBColor(0x00, 0x99, 0xff): "Non-haemophiliacs with HIV",
    RGBColor(0xda, 0x73, 0x26): "Activist Groups",
    RGBColor(0xc4, 0x59, 0x11): "Activist Groups",
    RGBColor(0x00, 0xb0, 0x50): "Treatments",
    RGBColor(0x33, 0xcc, 0x33): "Treatments",
    RGBColor(0x37, 0x46, 0xd1): "Campaigns",
    RGBColor(0x44, 0x72, 0xc4): "Campaigns",
    RGBColor(0x00, 0x70, 0xc0): "Campaigns",
    RGBColor(0x2e, 0x13, 0xf9): "Campaigns",
    RGBColor(0x00, 0x00, 0xff): "Campaigns",
    RGBColor(0x00, 0x33, 0xcc): "Campaigns",
    RGBColor(0x21, 0x4b, 0xff): "Campaigns",
    RGBColor(0x06, 0x30, 0xe0): "Campaigns",
    RGBColor(0x0e, 0x51, 0xe8): "Campaigns",
    RGBColor(0xee, 0x00, 0x00): "Concerning Haemophiliacs",
    RGBColor(0xff, 0x00, 0x00): "Concerning Haemophiliacs",
    RGBColor(0xc0, 0x00, 0x00): "HIV/AIDs issues",
    RGBColor(0x8a, 0x21, 0xab): "Events",
    RGBColor(0xdd, 0x5d, 0xd4): "Events",
    RGBColor(0xb1, 0x25, 0xa7): "Funds, inquires or legal cases",
    RGBColor(0x90, 0x78, 0xf8): "Funds, inquires or legal cases",
    RGBColor(0x05, 0xff, 0x76): "Science",
    RGBColor(0x66, 0xff, 0xcc): "Science",
    RGBColor(0x00, 0xff, 0x99): "Science",
    RGBColor(0x0f, 0x4d, 0xb1): "Ireland",
    RGBColor(0x48, 0xb8, 0xab): "Ireland",
    WD_COLOR_INDEX.YELLOW: "Specific reference to Hepatitis C",
    RGBColor(0x12, 0x12, 0x12): "No topic"
}

def extract_only_bullets(docx_path):
    doc = Document(docx_path)
    bullet_points = []
    
    for paragraph in doc.paragraphs:
        pPr = paragraph._element.pPr
        
        if pPr is not None:
            numPr = pPr.find('.//{http://schemas.openxmlformats.org/wordprocessingml/2006/main}numPr')
            
            if numPr is not None:
                # Get the numId
                numId = numPr.find('.//{http://schemas.openxmlformats.org/wordprocessingml/2006/main}numId')
                
                if numId is not None:
                    num_id = numId.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val')
                    
                    # Check if it's a bullet (not numbered)
                    if is_bullet_list(doc, num_id):
                        topics_incom = {color_mapping[run.font.color.rgb] for run in paragraph.runs if run.font.color.rgb is not None and run.font.color.rgb != RGBColor(0x12, 0x12, 0x12)}
                        rest_topics = {run.font.highlight_color for run in paragraph.runs}
                        if rest_topics == {None}:
                            topics = topics_incom
                        else:
                            topics = topics_incom | rest_topics
                            topics.discard(None) 
                        bullet_points.append((paragraph.text, topics))
    
    return bullet_points

def is_bullet_list(doc, num_id):
    """Check if a numbering ID corresponds to a bullet list"""
    try:
        numbering_part = doc.part.numbering_part
        numbering_xml = numbering_part.element
        
        # Find the num element with this numId
        num = numbering_xml.find(f'.//{{{numbering_xml.nsmap["w"]}}}num[@{{{numbering_xml.nsmap["w"]}}}numId="{num_id}"]')
        
        if num is not None:
            # Get the abstractNumId
            abstractNumId = num.find(f'.//{{{numbering_xml.nsmap["w"]}}}abstractNumId')
            abstract_num_id = abstractNumId.get(f'{{{numbering_xml.nsmap["w"]}}}val')
            
            # Find the abstractNum definition
            abstractNum = numbering_xml.find(f'.//{{{numbering_xml.nsmap["w"]}}}abstractNum[@{{{numbering_xml.nsmap["w"]}}}abstractNumId="{abstract_num_id}"]')
            
            if abstractNum is not None:
                # Check the first level's numFmt
                lvl = abstractNum.find(f'.//{{{numbering_xml.nsmap["w"]}}}lvl[@{{{numbering_xml.nsmap["w"]}}}ilvl="0"]')
                numFmt = lvl.find(f'.//{{{numbering_xml.nsmap["w"]}}}numFmt')
                
                # Bullet lists use 'bullet' format, numbered use 'decimal', 'lowerLetter', etc.
                fmt_val = numFmt.get(f'{{{numbering_xml.nsmap["w"]}}}val')
                return fmt_val == 'bullet'
    except:
        pass
    
    return False

In [3]:
bullets = extract_only_bullets("Timetable-REV5 Hypotheses re press coverage of Hepatitis C in UK.docx")
bullets

[('1950: The Haemophilia Society (UK) was formed.', {'Activist Groups'}),
 ('Late 1980s: Early Treatments: Interferon was the first antiviral treatment for hepatitis C.  significant side effects and lower cure rates.',
  {'Treatments'}),
 ('Victims and their families first started calling for compensation, often on the grounds of medical negligence, as more evidence is gathered about link between blood products and HIV/hepatitis infections',
  set()),
 ('1987. The Birchgrove Group (often referred to as the BirchGrove Society in contexts of support), which supports people with haemophilia, HIV, and Hepatitis C, established in Wales by a small group of patients at the University Hospital of Wales (UHW) in Cardiff.  Early 1990s expanded to a national level. ',
  {'Activist Groups', 'Concerning Haemophiliacs'}),
 ('November 1987: announcement of £10 million payment for haemophiliacs infected with HIV',
  {'Campaigns', 'Concerning Haemophiliacs', 'HIV/AIDs issues'}),
 ('7 March 1988: Britis

In [4]:
import re
from collections import defaultdict

months = r'(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)'

date_pattern = r'\b(?:' \
    rf'\d{{1,2}}\s+{months}\s+\d{{4}}|' \
    rf'{months}\s+\d{{1,2}},?\s+\d{{4}}|' \
    r'(?:Early|Mid|Late)\s+\d{4}s?|' \
    rf'{months}\s+\d{{4}}' \
    r')\b'

def extract_date(text):
    """Extract the date string from text"""
    match = re.search(date_pattern, text, re.IGNORECASE)
    if match:
        return match.group(0)
    return None

def remove_date(text, date_str):
    """Remove the date from the text and clean up extra whitespace"""
    cleaned = text.replace(date_str, '').strip()
    # Clean up any double spaces or extra punctuation left behind
    cleaned = re.sub(r'\s+', ' ', cleaned)
    cleaned = re.sub(r'^\s*[-:]\s*', '', cleaned)  # Remove leading dash or colon
    return cleaned

def group_bullets_by_date_dict(bullet_list):
    """Create dictionary with dates as keys and associated text as values"""
    date_dict = defaultdict(list)
    topic_dict = {}
    current_date = None
    
    for bullet in bullet_list:
        text, topics = bullet
        date_str = extract_date(text)
        
        if date_str:
            # This bullet has a date - start a new entry
            current_date = date_str
            # Remove date from text and make it the first element
            text_without_date = remove_date(text, date_str)
            date_dict[current_date].append(text_without_date)
        else:
            # This bullet doesn't have a date - add to current date's list
            if current_date:
                date_dict[current_date].append(text)
        topic_dict[date_str] = topics
        
    
    return dict(date_dict), topic_dict


In [5]:
grouped, topics = group_bullets_by_date_dict(bullets)
grouped, topics, len(grouped), len(topics)

({'Late 1980s': ['Early Treatments: Interferon was the first antiviral treatment for hepatitis C. significant side effects and lower cure rates.',
   'Victims and their families first started calling for compensation, often on the grounds of medical negligence, as more evidence is gathered about link between blood products and HIV/hepatitis infections'],
  'Early 1990s': ['1987. The Birchgrove Group (often referred to as the BirchGrove Society in contexts of support), which supports people with haemophilia, HIV, and Hepatitis C, established in Wales by a small group of patients at the University Hospital of Wales (UHW) in Cardiff. expanded to a national level.'],
  'November 1987': ['announcement of £10 million payment for haemophiliacs infected with HIV'],
  '7 March 1988': ['British Liver Trust incorporated and established. (Founded by hepatologist Professor Dame Sheila Sherlock, leading charity for adults, focusing on raising awareness, providing support, and campaigning for improve

In [6]:
topics

{None: {'Concerning Haemophiliacs'},
 'Late 1980s': {'Treatments'},
 'Early 1990s': {'Activist Groups', 'Concerning Haemophiliacs'},
 'November 1987': {'Campaigns', 'Concerning Haemophiliacs', 'HIV/AIDs issues'},
 '7 March 1988': {'Activist Groups'},
 '10 March 1988': {'Campaigns', 'Funds, inquires or legal cases'},
 'April 21, 1989': {'Science'},
 'July 1988': {'Campaigns', 'HIV/AIDs issues', 'Non-haemophiliacs with HIV'},
 'June 1989': {'Campaigns',
  'Concerning Haemophiliacs',
  'Funds, inquires or legal cases',
  'HIV/AIDs issues'},
 '29 January 1990': {'Campaigns',
  'Concerning Haemophiliacs',
  'Funds, inquires or legal cases',
  'Ireland'},
 'April 1991': {'Campaigns',
  'Concerning Haemophiliacs',
  'Funds, inquires or legal cases',
  'HIV/AIDs issues'},
 '3 May 1991': {'Campaigns',
  'Funds, inquires or legal cases',
  'HIV/AIDs issues',
  'Ireland'},
 'Sept 1991': {'Events', 'Science', <WD_COLOR_INDEX.YELLOW: 7>},
 '17 February 1992': {'Campaigns', 'Non-haemophiliacs with H

# Retrieval

In [6]:
import pandas as pd

df = pd.read_csv("./all_ratings811.csv")
def parse_date(val):
    for fmt in ('%Y-%m-%d', '%B %d, %Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError) as e:
            continue
    return pd.NaT  # fallback if nothing matches

df['Date'] = df['Date'].apply(lambda x: x.replace(" 00:00:00", "")).apply(parse_date)
df = df[["Title", "Date", "Author", "Newspaper", "Source", "newspaper_type", "chunks"]]

In [7]:
df

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks
0,International Companies: Technology - Pounds 2...,1983-02-11,DAVID FISHLOCK,Financial Times,Factiva - HepC,broadsheet,BRITAIN'S biotechnologists are to have a new P...
1,Study confirms fears on spread of Aids,1984-08-31,ANDREW VEITCH,The Guardian,Lexis-Nexis - Times,broadsheet,Fears that the Aids virus is widespread among ...
2,Maternity unit where waste parades the corrido...,1984-10-20,No author in byline,The Guardian,Lexis-Nexis - Times,broadsheet,Dawn in the Rosie maternity unit at the 900-be...
3,Prisons inspector condemns sanitary conditions...,1984-10-24,AILEEN BALLANTYNE,The Guardian,Lexis-Nexis - Times,broadsheet,Gaol conditions fall short of the standard exp...
4,Financial Times Survey: China - Business - Gro...,1984-10-29,No author in byline,Financial Times,Factiva - HepC,broadsheet,CLIMATENorth China (including Beijing) - Winte...
...,...,...,...,...,...,...,...
7562,Monday \nEdition 1; \nNational Edition\n\nPat...,2016-06-13,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 13, 2016 Monday \n..."
7563,Saturday \nEdition 2; \nNational Edition\n\nD...,2016-08-06,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 6, 2016 Saturday ..."
7564,Wednesday \nEdition 2; \nNational Edition\n\n...,2016-10-05,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 5, 2016 Wednesda..."
7565,Friday \nEdition 1; \nIreland\n\nclassical Le...,2016-10-14,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 14, 2016 Friday ..."


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("/Users/pnadel01/fp_models/google_embeddinggemma-300m", device="mps")

In [8]:
# test key: '29 August 2003'

sent_sim_template = "task: sentence similarity | query: {content}"

# date_key_from_timetable = '29 August 2003'
# associated_text = grouped[date_key_from_timetable]
# query_embedding = model.encode(sent_sim_template.format(content=associated_text))
# query_embedding.shape

In [15]:
times = df[df.Newspaper == "The Times"]
times

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks
7,Travel: How to stay healthy on your summer bre...,1985-07-06,ISABEL WALKER,The Times,Factiva - HepC,broadsheet,Holidays are one of the 30 most stressful even...
8,Family Money: Taking the sting from jabs / Ino...,1985-08-03,VIVIEN GOLDSMITH,The Times,Factiva - HepC,broadsheet,The series of jabs you will need if you are of...
9,A BREAKDOWN OF THE BREAKDOWNS (732),1985-08-14,JOHN GOODBODY,The Times,Factiva - HepC,broadsheet,Steve Cram1981Has a winter of injuries.August:...
10,Times Diary: Les-majeste (105),1985-08-30,PHS,The Times,Factiva - HepC,broadsheet,After years of pretending to find Barry Humphr...
13,Books: Travelling pros(e) on the exotic job / ...,1985-11-28,ALLAN MASSIE,The Times,Factiva - HepC,broadsheet,Muddling Through In Madagascar by Dervla Murph...
...,...,...,...,...,...,...,...
7562,Monday \nEdition 1; \nNational Edition\n\nPat...,2016-06-13,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 13, 2016 Monday \n..."
7563,Saturday \nEdition 2; \nNational Edition\n\nD...,2016-08-06,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 6, 2016 Saturday ..."
7564,Wednesday \nEdition 2; \nNational Edition\n\n...,2016-10-05,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 5, 2016 Wednesda..."
7565,Friday \nEdition 1; \nIreland\n\nclassical Le...,2016-10-14,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 14, 2016 Friday ..."


In [16]:
key_dt = pd.to_datetime(date_key_from_timetable)
buffer = pd.DateOffset(years=1)

zone_of_interest = times[
    (times.Date >= key_dt - buffer) & # backward looking
    (times.Date <= key_dt + buffer)   # forward looking
]
zone_of_interest 

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks
1951,Lawsuit reassurances fail to revive Weston - M...,2002-08-31,No author in byline,The Times,Factiva - HepC,broadsheet,REASSURING comments that Weston Medical's need...
1954,Body piercing craze 'is putting teenagers' hea...,2002-09-10,Sam Lister and Patrick Barkham.,The Times,Factiva - HepC,broadsheet,"Body piercings below the neck, once the preser..."
1955,Gilt of the body politic - Cover story.,2002-09-11,Theodore Dalrymple.,The Times,Factiva - HepC,broadsheet,As tough new restrictions are proposed on body...
1956,The Times Vet - Pets.,2002-09-14,James Allcock.,The Times,Factiva - HepC,broadsheet,Q You answered a question about leg amputation...
1960,Liver failure death due to Ecstasy - In brief.,2002-09-17,No author in byline,The Times,Factiva - HepC,broadsheet,A 14-year-old girl suffered severe liver failu...
...,...,...,...,...,...,...,...
7433,", Saturday\n\n Shopping trips",2004-06-12,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 12, 2004, Saturday\..."
7434,", Friday\n\n What's Up, Doc?",2004-06-18,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 18, 2004, Friday\n\..."
7435,", Thursday\n\n So, anything to declare?",2004-07-29,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJuly 29, 2004, Thursday\..."
7436,", Tuesday\n\n Could meat give you Alzheimer's?",2004-08-10,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 10, 2004, Tuesday..."


In [17]:
doc_texts = [t.Title + "\n" + t.chunks for t in zone_of_interest.itertuples()]
doc_embeddings = model.encode([sent_sim_template.format(content=text) for text in doc_texts], show_progress_bar=True, device="mps")
doc_embeddings.shape

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

(121, 768)

In [18]:
sim_scores = doc_embeddings @ query_embedding.T
rel_ids_and_scores =[(i, sim_scores[i]) for i in (sim_scores).argsort()[::-1]]
rel_ids_and_scores

[(np.int64(65), np.float32(0.7566832)),
 (np.int64(50), np.float32(0.7512808)),
 (np.int64(112), np.float32(0.750149)),
 (np.int64(49), np.float32(0.7292455)),
 (np.int64(22), np.float32(0.71006584)),
 (np.int64(94), np.float32(0.68818104)),
 (np.int64(12), np.float32(0.67302406)),
 (np.int64(55), np.float32(0.6703769)),
 (np.int64(26), np.float32(0.66509414)),
 (np.int64(30), np.float32(0.657603)),
 (np.int64(98), np.float32(0.64630866)),
 (np.int64(92), np.float32(0.6392649)),
 (np.int64(66), np.float32(0.63884985)),
 (np.int64(20), np.float32(0.63880205)),
 (np.int64(27), np.float32(0.6387496)),
 (np.int64(16), np.float32(0.6191523)),
 (np.int64(97), np.float32(0.6155448)),
 (np.int64(75), np.float32(0.60481125)),
 (np.int64(38), np.float32(0.60325783)),
 (np.int64(59), np.float32(0.59671074)),
 (np.int64(107), np.float32(0.59583867)),
 (np.int64(68), np.float32(0.5955117)),
 (np.int64(33), np.float32(0.5744972)),
 (np.int64(41), np.float32(0.5735586)),
 (np.int64(21), np.float32(0.

In [19]:
import numpy as np

def scale_score(sim_scores, item_dates, target_date, scaler_func, date_weight):
    sim_scores = np.array(sim_scores)
    days_diffs = np.abs(np.array([(target_date - d).days for d in item_dates]))
    date_scores = np.array([scaler_func(d) for d in days_diffs])
    composite = (1 - date_weight) * sim_scores + date_weight * date_scores
    return composite, date_scores

def inverse_square(days_diff, max_days=365*2):
    normalized_diff = days_diff / max_days
    return 1 / (1 + normalized_diff ** 2)

In [20]:
composite, date_scores = scale_score(sim_scores, zone_of_interest.Date, key_dt, inverse_square, 0.5)
composite, date_scores 

(array([0.63313717, 0.60922786, 0.55187218, 0.62564215, 0.61986936,
        0.64671886, 0.61361636, 0.60399444, 0.56903819, 0.60345983,
        0.6917313 , 0.68657356, 0.76632209, 0.61110959, 0.66642766,
        0.59193776, 0.75633707, 0.72955954, 0.72195708, 0.62931908,
        0.77566978, 0.74199704, 0.81648133, 0.65916146, 0.6118918 ,
        0.66339324, 0.80328683, 0.79562996, 0.66136626, 0.71856554,
        0.80696516, 0.7417867 , 0.71299678, 0.77255995, 0.70150345,
        0.71523417, 0.6815965 , 0.74144122, 0.78972555, 0.71902697,
        0.69361535, 0.77896202, 0.67501675, 0.77019896, 0.70182789,
        0.75365121, 0.72516232, 0.72241352, 0.73512606, 0.8646218 ,
        0.87563945, 0.77069197, 0.71821433, 0.75197838, 0.71416696,
        0.83422951, 0.7531749 , 0.7104225 , 0.70514935, 0.79465894,
        0.70990356, 0.67314103, 0.73134059, 0.75861481, 0.72596642,
        0.85860128, 0.7986481 , 0.68546126, 0.77288998, 0.74223986,
        0.60553774, 0.70168495, 0.70822513, 0.73

In [21]:
from pprint import pprint

i = composite.argsort()[::-1][0]

print(i)
print(composite[i])
print(zone_of_interest.iloc[i].Title)
print(zone_of_interest.iloc[i].Date)
pprint(zone_of_interest.iloc[i].chunks)
print()
pprint(associated_text)

50
0.8756394540428893
Cash hopes for hepatitis victims
2003-08-30 00:00:00
('THE campaign to win compensation payments for Scots infected with hepatitis '
 'C from infected NHS blood products took an important step forward yesterday '
 'as the Scottish Executive announced that the UK Government had agreed that '
 'such payments could be made by the devolved body.Malcolm Chisholm, '
 "Scotland's Health Minister, said that a similar scheme was being established "
 'in England by the Department of Health.A stumbling block to implementing any '
 'such scheme remains, however, because the Department of Work and Pensions in '
 'Whitehall has given no guarantee that it will not claw back social security '
 'payments from hepatitis victims given compensation.Under the proposed scheme '
 'in Scotland, ex gratia payments of Pounds 20,000 would be made to all those '
 'who have the hepatitis C virus as a result of infected NHS blood products. A '
 'further Pounds 25,000 would go to to those in an

# Expand to dataset

In [27]:
from pprint import pprint
def time_bounded_search(df, model, date_key_from_timetable, month_threshold=2, go_backwards=True, scale_fn=inverse_square, date_weight=0.5, sim_threshold=.75, newspaper=None):
    if newspaper:
        df = df[df.Newspaper == newspaper]
        print(f"Filtered dataframe to {len(df)} documents from newspaper: {newspaper}")
    associated_text = grouped[date_key_from_timetable]
    query_embedding = model.encode(sent_sim_template.format(content=associated_text))

    key_dt = pd.to_datetime(date_key_from_timetable)
    buffer = pd.DateOffset(months=month_threshold)

    if go_backwards:
        zone_of_interest = df[
            (df.Date >= key_dt - buffer) & # backward looking
            (df.Date <= key_dt + buffer)   # forward looking
        ]
    else:   
        zone_of_interest = df[
            (df.Date >= key_dt) &          # backward looking
            (df.Date <= key_dt + buffer)   # forward looking
        ]

    print(f"Processing {len(zone_of_interest)} documents in the zone of interest...")
    doc_texts = [t.Title + "\n" + t.chunks for t in zone_of_interest.itertuples()]
    doc_embeddings = model.encode([sent_sim_template.format(content=text) for text in doc_texts], show_progress_bar=True, device="mps")
    sim_scores = doc_embeddings @ query_embedding.T
    composite, _ = scale_score(sim_scores, zone_of_interest.Date, key_dt, scale_fn, date_weight)

    results = []
    for i in composite.argsort()[::-1]:
        score = composite[i]
        if score >= sim_threshold:
            results.append(zone_of_interest.iloc[i].to_dict())
    return results

def report_result(result, date_key_from_timetable):
    print("-"*40)
    print(f"Date from timetable: {date_key_from_timetable}")
    print(f"Text from timetable:")
    pprint(grouped[date_key_from_timetable])
    print("-"*40)
    print(f"Title: {result['Title']}")
    print(f"Date: {result['Date']}")
    print(f"Newspaper: {result['Newspaper']}")
    pprint(f"{result['chunks']}")
    # print(f"Rating: {result['rating']*5}")
    print("-"*40)

In [28]:
grouped.keys()

dict_keys(['Late 1980s', 'Early 1990s', 'November 1987', '7 March 1988', '10 March 1988', 'April 21, 1989', 'July 1988', 'June 1989', '29 January 1990', 'April 1991', '3 May 1991', 'Sept 1991', '17 February 1992', '27 April 1992', '29 March 1993', 'May 1993', '11 July 1995', '26 March 2001', 'August 2001', 'June 2002', '20 January 2003', '29 August 2003', 'January 2004', '25 March 2004', 'January 2006', 'July 2006', '19 February 2007', 'March 2007', '23 April 2008', 'February 2009', 'March 2009', '20 May 2009', 'April 2010', '14 October 2010', 'January 2011', '8 March 2011', '22 June 2011', 'May 2011', 'October 2011', 'November 2013', 'December 2013', 'April 2014', 'August 2014', 'October 2014', 'March 2015', '25 March 2015', 'October 2015', 'November 2015', 'June 2016', 'January 2017', '11 July 2017', 'August 2017', 'May 2018', '2 July 2018', '5 October 2020', '10 December 2020', '20 May 2024', '31 March 2025', '9 July 2025', '21 July 2025', '30 October 2025', 'November 2025', '31 Dec

In [45]:
date_key = 'Late 1980s'

results = time_bounded_search(
    df,
    model,
    date_key_from_timetable=date_key,
    go_backwards=False,
    month_threshold=12,
    newspaper="The Times"
)

Filtered dataframe to 1146 documents from newspaper: The Times


DateParseError: Unknown datetime string format, unable to parse: Late 1980s

In [34]:
len(results)

2

In [37]:
report_result(results[0], date_key)

----------------------------------------
Date from timetable: April 21, 1989
Text from timetable:
['‘Discovery’/identification of HCV - two seminal papers were published in the '
 'journal Science.',
 'Development of the first antibody test (an ELISA to detect anti-C100-3 '
 'antibodies) by Chiron Corporation, partnering with the CDC']
----------------------------------------
Title: Hepatitis screening is considered;Blood transfusions
Date: 1989-08-25 00:00:00
Newspaper: The Times
('A test that can detect a recently discovered type of hepatitis virus may be '
 'used to screen all blood donations (Our Science Correspondent writes).The '
 'National Blood Transfusion Service is carrying out studies on the '
 'effectiveness of the test, developed in the United States, for hepatitis C, '
 'usually a milder form of the liver disease.The prevalence of the infection '
 'is unknown in Britain, but it is the most common form of hepatitis occurring '
 'after blood transfusions in some developing 

In [3]:
df

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks
0,International Companies: Technology - Pounds 2...,1983-02-11,DAVID FISHLOCK,Financial Times,Factiva - HepC,broadsheet,BRITAIN'S biotechnologists are to have a new P...
1,Study confirms fears on spread of Aids,1984-08-31,ANDREW VEITCH,The Guardian,Lexis-Nexis - Times,broadsheet,Fears that the Aids virus is widespread among ...
2,Maternity unit where waste parades the corrido...,1984-10-20,No author in byline,The Guardian,Lexis-Nexis - Times,broadsheet,Dawn in the Rosie maternity unit at the 900-be...
3,Prisons inspector condemns sanitary conditions...,1984-10-24,AILEEN BALLANTYNE,The Guardian,Lexis-Nexis - Times,broadsheet,Gaol conditions fall short of the standard exp...
4,Financial Times Survey: China - Business - Gro...,1984-10-29,No author in byline,Financial Times,Factiva - HepC,broadsheet,CLIMATENorth China (including Beijing) - Winte...
...,...,...,...,...,...,...,...
7562,Monday \nEdition 1; \nNational Edition\n\nPat...,2016-06-13,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nJune 13, 2016 Monday \n..."
7563,Saturday \nEdition 2; \nNational Edition\n\nD...,2016-08-06,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nAugust 6, 2016 Saturday ..."
7564,Wednesday \nEdition 2; \nNational Edition\n\n...,2016-10-05,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 5, 2016 Wednesda..."
7565,Friday \nEdition 1; \nIreland\n\nclassical Le...,2016-10-14,No author found,The Times,Lexis Nexus,broadsheet,"The Times (London)\n\nOctober 14, 2016 Friday ..."


In [9]:
doc_embeddings = model.encode([sent_sim_template.format(content=text) for text in df.chunks], show_progress_bar=True, device="mps")

Batches:   0%|          | 0/237 [00:00<?, ?it/s]

In [11]:
import numpy as np
np.save('doc-embeddings811.npy', doc_embeddings)

In [ ]:
doc_embeddings.shape

(7567, 768)

In [ ]:
doc_embeddings[zone_of_interest.index].shape

(121, 768)

In [ ]:
zone_of_interest.shape

(121, 7)

# Full

In [1]:
import json
from tbs import TimeTable, report_result, date_pattern
from pprint import pprint

timetable = TimeTable("Timetable-REV5 Hypotheses re press coverage of Hepatitis C in UK.docx", date_pattern)
with open("tbs_results825.json", "r") as json_file:
    results_by_date_key = json.load(json_file)

len(timetable.grouped), len(results_by_date_key)

(112, 112)

In [2]:
pprint(list(timetable.grouped.keys()))

['Late 1980s_1',
 'Late 1980s_2',
 'Early 1990s_1',
 'November 1987_1',
 '7 March 1988_1',
 '10 March 1988_1',
 'April 21, 1989_1',
 'April 21, 1989_2',
 'July 1988_1',
 'June 1989_1',
 '29 January 1990_1',
 'April 1991_1',
 '3 May 1991_1',
 'Sept 1991_1',
 '17 February 1992_1',
 '27 April 1992_1',
 '29 March 1993_1',
 'May 1993_1',
 'May 1993_2',
 '11 July 1995_1',
 '26 March 2001_1',
 'August 2001_1',
 'August 2001_2',
 'June 2002_1',
 'June 2002_2',
 '20 January 2003_1',
 '29 August 2003_1',
 'January 2004_1',
 'September 1991_1',
 '25 March 2004_1',
 '25 March 2004_2',
 'January 2006_1',
 'July 2006_1',
 '19 February 2007_1',
 'March 2007_1',
 '23 April 2008_1',
 'February 2009_1',
 'March 2009_1',
 '20 May 2009_1',
 '20 May 2009_2',
 'April 2010_1',
 '14 October 2010_1',
 'January 2011_1',
 'January 2011_2',
 '8 March 2011_1',
 '22 June 2011_1',
 '22 June 2011_2',
 'May 2011_1',
 'October 2011_1',
 'October 2011_2',
 'November 2013_1',
 'December 2013_1',
 'April 2014_1',
 'August

In [3]:
for key in list(timetable.grouped.keys()):
    if len(timetable.grouped[key]) > 1:
        print(key)

In [4]:
import pandas as pd
import re

df = pd.read_csv("./all_ratings811.csv")
def parse_date(val):
    for fmt in ('%Y-%m-%d', '%B %d, %Y'):
        try:
            return pd.to_datetime(val, format=fmt)
        except (ValueError, TypeError) as e:
            continue
    return pd.NaT  # fallback if nothing matches

df['Date'] = df['Date'].apply(lambda x: x.replace(" 00:00:00", "")).apply(parse_date)
df = df[["Title", "Date", "Author", "Newspaper", "Source", "newspaper_type", "chunks"]]
df[df.chunks.apply(lambda x: len(re.findall(r"[M-m]avyret", x))) > 0]

,Title,Date,Author,Newspaper,Source,newspaper_type,chunks
5183,Non-profit’s $300 hepatitis C cure as effect...,2018-04-12,Julia Kollewe,The Guardian,Semi automated,broadsheet,Exclusive: 71 million people stand to benefit ...
5390,"Washington man, 49, willingly receives a heart...",2019-01-17,Mary Kekatos Health Reporter For Dailymail.com,The Daily Mail,Semi automated,tabloid,"* Kerry Hayes, 49, from Anacortes, Washington,..."
6517,The sick reality: Drugs advertised most on TV ...,2023-02-07,Cassidy Morrison Senior Health Reporter For Da...,The Daily Mail,Semi automated,tabloid,* At least 68 percent of top-selling drugs in ...


In [5]:
pprint(df.iloc[6517].Title)

('The sick reality: Drugs advertised most on TV have LEAST health benefits, '
 'according to analysis of $6bn-a-year industry')


In [6]:
pprint(df.iloc[6517].chunks)

("* At least 68 percent of top-selling drugs in 2020 are listed as 'low-added "
 "benefit' * Low-benefit drugmakers spent an average 14.3 percentage points "
 'more on ads * Direct to consumer advertising climbed from $1.3 bn in 1996 to '
 '$6 bn in 2020 The drugs that are advertised most on TV are also the least '
 'effective, according to a major analysis. Johns Hopkins University '
 'researchers found that spending on promoting prescription drugs rose from '
 '$1.3 billion in 1996 to $6 billion in 2016. Ninety-two of 135 drugs included '
 'in the study, or nearly 70 percent, were deemed to have little health '
 'benefits. Dr Michael DiStefano, the lead author of the study, said: The '
 'findings suggest that shifting promotional dollars to direct-to-consumer '
 'advertising potentially reflects a strategy to drive patient demand for '
 'drugs that clinicians would be less likely to prescribe. When a consumer '
 'sees these advertisements on TV or social media, they should really que

In [7]:
date_key = 'August 2017_1' 
result = results_by_date_key[date_key]
report_result(result, date_key, timetable.grouped, timetable.topics)

----------------------------------------
Date from timetable: August 2017_1
['Mavyret (glecaprevir/pibrentasvir) approved']
'Topics: set()'
----------------------------------------
Found 6 results
Text from timetable:
**Relevant Article 1**
Title: Hepatitis C 'cure' drug approved by NHS
Date: 2017-01-17 00:00:00
Newspaper: Metro
("IN BRIEFA MEDICINE giving the 'prospect of a cure' for Hepatitis C has been "
 'accepted for use by NHS Scotland.The Scottish Medicines Consortium approved '
 "Elbasvirgrazoprevir, known as Zepatier, on Monday, saying it is 'a "
 'combination of two medicines that inhibit the replication of the Hepatitis C '
 "virus and offers the prospect of a cure for Hepatitis C infection'.The "
 'consortium also approved Pembrolizumab, known as Keytruda, which can give '
 'latestage non-small cell lung cancer patients two more months to live.')
----------------------------------------
**Relevant Article 2**
Title: Jury orders Gilead to pay Merck $2.54bn in damages in pate

In [8]:
for date_key in timetable.grouped.keys():
    result = results_by_date_key[date_key]
    if len(result) == 0:
        print(date_key)
        pprint(timetable.grouped[date_key])
        print("-"*40)

10 March 1988_1
['Macfarlane Trust established']
----------------------------------------
April 21, 1989_1
['‘Discovery’/identification of HCV - two seminal papers were published in the '
 'journal Science.']
----------------------------------------
April 21, 1989_2
['Development of the first antibody test (an ELISA to detect anti-C100-3 '
 'antibodies) by Chiron Corporation, partnering with the CDC']
----------------------------------------
July 1988_1
['Civil service starts discussing giving money to non-haemophiliacs with HIV']
----------------------------------------
29 January 1990_1
['Macfarlane I Special Payments Trust established (MFTSP1): £20,000 to each '
 'haemophiliac (or bereaved families)']
----------------------------------------
April 1991_1
['£42 million ex gratia payment announced to end the HIV haemophiliacs '
 'litigation. Case settled.']
----------------------------------------
3 May 1991_1
['Macfarlane Trust Special Payment II (MFTSP2): Lump sums to 1,437 people '

In [9]:
for date_key in timetable.grouped.keys():
    result = results_by_date_key[date_key]
    if len(result) > 2 and len(result) > 0:
        print(date_key)
        pprint(timetable.grouped[date_key])
        print("-"*40)

Early 1990s_1
['1987. The Birchgrove Group (often referred to as the BirchGrove Society in '
 'contexts of support), which supports people with haemophilia, HIV, and '
 'Hepatitis C, established in Wales by a small group of patients at the '
 'University Hospital of Wales (UHW) in Cardiff. expanded to a national level.']
----------------------------------------
November 1987_1
['announcement of £10 million payment for haemophiliacs infected with HIV']
----------------------------------------
29 March 1993_1
['Eileen Trust established. £500,000 for non-haemophiliacs with HIV. Also had '
 'to sign waivers.']
----------------------------------------
May 1993_2
['1994 to 1998 Irish HCV scandal**']
----------------------------------------
11 July 1995_1
['. Funding for a look-back exercise to identify any patients who may have '
 'been infected with HCV']
----------------------------------------
26 March 2001_1
['A and Others vs. National Blood Authority, concerned only hepatitis C. Case '


# Timeline

In [10]:
import pandas as pd
events = pd.DataFrame.from_dict(timetable.grouped, orient="index").reset_index().rename(columns={"index":"timetable_entry", 0:"description"})
topics = pd.DataFrame.from_dict({k:", ".join(v) for k,v in timetable.topics.items()}, orient="index").reset_index().rename(columns={"index":"timetable_entry", 0:"topics"})
df = events.merge(topics, on="timetable_entry")[:77] # getting rid of extra data at the bottom...
df

,timetable_entry,description,topics
0,Late 1980s_1,Early Treatments: Interferon was the first ant...,Treatments
1,Late 1980s_2,Victims and their families first started calli...,
2,Early 1990s_1,1987. The Birchgrove Group (often referred to ...,"Concerning Haemophiliacs, Activist Groups"
3,November 1987_1,announcement of £10 million payment for haemop...,"HIV/AIDs issues, Campaigns, Concerning Haemoph..."
4,7 March 1988_1,British Liver Trust incorporated and establish...,Activist Groups
...,...,...,...
72,21 July 2025_1,The Infected Blood Compensation Authority and ...,"Funds, inquires or legal cases, Campaigns"
73,30 October 2025_1,The Government laid a third set of regulations...,"Funds, inquires or legal cases, Campaigns"
74,November 2025_1,A public consultation was launched on more com...,"Funds, inquires or legal cases, Campaigns"
75,31 December 2025_1,Third infected blood compensation scheme regul...,"Funds, inquires or legal cases, Campaigns"


In [11]:
import re 

months = r'(?:January|February|March|April|May|June|July|August|September|October|November|December|Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)'
date_pattern = r'\b(?:' \
    rf'\d{{1,2}}\s+{months}\s+\d{{4}}|' \
    rf'{months}\s+\d{{1,2}},?\s+\d{{4}}|' \
    r'(?:Early|Mid|Late)\s+\d{4}s?|' \
    rf'{months}\s+\d{{4}}' \
    r')\b'

def extract_date(text):
    """Extract the date string from text"""
    match = re.search(date_pattern, text, re.IGNORECASE)
    if match:
        return match.group(0)
    return None

def handle_ambiguous_dates(date_str):
    year = re.findall(r"(\d{4})", date_str)[0]
    if "Late" in date_str:
        year = int(year)+6
        new_date_str = f"1/1/{year}"
    elif "Early" in date_str:
        new_date_str = f"1/1/{year}"
    else:
        new_date_str = extract_date(date_str)
    return pd.to_datetime(new_date_str)

def handle_ambiguous_dates_end(date_str):
    year = re.findall(r"(\d{4})", date_str)[0]
    if "Late" in date_str:
        year = int(year)+6
        new_date_str = f"1/1/{year+3}"
        return pd.to_datetime(new_date_str) 
    elif "Early" in date_str:
        new_date_str = f"1/1/{int(year)+5}"
        return pd.to_datetime(new_date_str) 
    else:
        new_date_str = extract_date(date_str)
        return None

df["start"] = df["timetable_entry"].apply(lambda x: handle_ambiguous_dates(x.split("_")[0]))
df["end"] = df["timetable_entry"].apply(lambda x: handle_ambiguous_dates_end(x.split("_")[0]))

In [12]:
results_by_date_key

{'Late 1980s_1': [{'Title': "\nThe Department of Health has approved the marketing of Wellferon, Wellcome's anti-cancer drug.\n",
   'Date': '1986-03-26 00:00:00',
   'Author': 'No author in byline',
   'Newspaper': 'The Evening Standard',
   'Source': 'Factiva - NANBH',
   'newspaper_type': 'tabloid',
   'chunks': ' Wellferon is based on interferon. It is now licensed for treatment of hairy cell leukaemia, a rare form of blood cancer. Wellcome is studying the effects of Wellferon on chronic hepatitis B.'}],
 'Late 1980s_2': [{'Title': "'Aids risk' in surgery rubbish",
   'Date': '1986-03-25 00:00:00',
   'Author': 'No author',
   'Newspaper': 'The Times',
   'Source': 'Newspaper site',
   'newspaper_type': 'broadsheet',
   'chunks': '\'Aids risk\' in surgery rubbish\n\nChildren were exposed to a "clinical cocktail of risk" including hepatitis and Aids from waste materials strewn outside a Bradford doctors\' surgery, city magistrates were told yesterday.\n\nA girl aged five had gone ho

In [30]:
results_by_date_key[df.iloc[2].timetable_entry]

[{'Title': 'Tests reveal liver disease treatment.',
  'Date': '1989-11-30 00:00:00',
  'Author': '    By Celia Hall, Medical Editor ',
  'Newspaper': 'The Independent',
  'Source': 'Semi automated',
  'newspaper_type': 'other',
  'chunks': ' Scientists in the United States have discovered a treatment for the liver disease believed to be responsible for 90 per cent of cases of hepatitis that follow blood transfusions. Half of the patients responded to treatment with interferon, some to low doses. Mild flu-like side effects disappeared quite quickly.'},
 {'Title': ', Thursday\n\nA to E of hepatitis; Medical Briefing',
  'Date': '1989-11-30 00:00:00',
  'Author': 'No author found',
  'Newspaper': 'The Times',
  'Source': 'Lexis Nexus',
  'newspaper_type': 'broadsheet',
  'chunks': "The Times (London)\n\nNovember 30 1989, Thursday\n\nA to E of hepatitis; Medical Briefing\n\nBYLINE: Dr THOMAS STUTTAFORD\n\nSECTION: Issue 63565.\n\nLENGTH: 634 words\n\n \n The bewitching smile with which the

In [39]:
long_days = 5
items = []
for i, row in df.iterrows():
    span = (row.end - row.start).days if row.end else 0
    item = {
        "id":i+1,
        "content":row["description"],
        "start":row["start"].strftime("%Y-%m-%dT%H:%M:%S"),
        "type": "range" if span >= long_days else "box",
        "detail": {
            "topics":row["topics"] if row["topics"] != "" else "No topic",
            "articles":results_by_date_key[row["timetable_entry"]] if results_by_date_key[row["timetable_entry"]] != "" else "No articles found for this event"
        }
    }
    if row["end"] is not pd.NaT:
        item["end"] = row["end"].strftime("%Y-%m-%dT%H:%M:%S")
    if row.end and span >= long_days:
        item["end"] = row.end.isoformat()
    items.append(item)
items[:5]

[{'id': 1,
  'content': 'Early Treatments: Interferon was the first antiviral treatment for hepatitis C. significant side effects and lower cure rates.',
  'start': '1986-01-01T00:00:00',
  'type': 'range',
  'detail': {'topics': 'Treatments',
   'articles': [{'Title': "\nThe Department of Health has approved the marketing of Wellferon, Wellcome's anti-cancer drug.\n",
     'Date': '1986-03-26 00:00:00',
     'Author': 'No author in byline',
     'Newspaper': 'The Evening Standard',
     'Source': 'Factiva - NANBH',
     'newspaper_type': 'tabloid',
     'chunks': ' Wellferon is based on interferon. It is now licensed for treatment of hairy cell leukaemia, a rare form of blood cancer. Wellcome is studying the effects of Wellferon on chronic hepatitis B.'}]},
  'end': '1989-01-01T00:00:00'},
 {'id': 2,
  'content': 'Victims and their families first started calling for compensation, often on the grounds of medical negligence, as more evidence is gathered about link between blood products

In [49]:
import anywidget, traitlets

class VisTimeline(anywidget.AnyWidget):
    _esm = """
    import { Timeline, DataSet } from "https://esm.sh/vis-timeline@7.7.3/standalone";

    function ensureStylesheet() {
      const id = "vis-timeline-css";
      if (document.getElementById(id)) return;
      const link = document.createElement("link");
      link.id = id;
      link.rel = "stylesheet";
      link.href = "https://esm.sh/vis-timeline@7.7.3/styles/vis-timeline-graph2d.min.css";
      document.head.appendChild(link);
    }

    const esc = (s) => String(s ?? "").replace(/[&<>"]/g,
      c => ({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;"}[c]));

    function render({ model, el }) {
      ensureStylesheet();
      el.classList.add("vt-widget");

      const container = document.createElement("div");
      const panel = document.createElement("div");
      panel.className = "vt-panel";
      el.appendChild(container);
      el.appendChild(panel);

      const defaults = {
        height: model.get("height"),
        verticalScroll: true,
        horizontalScroll: true,
        zoomKey: "ctrlKey",
        orientation: { axis: "top", item: "top" },
        stack: true,
        margin: { item: { horizontal: 10, vertical: 10 }, axis: 20 },
        tooltip: { followMouse: true, overflowMethod: "flip" },
      };
      const options = Object.assign(defaults, model.get("options"));

      const items = new DataSet(model.get("items"));
      const groups = model.get("groups");
      const tl = new Timeline(
        container, items,
        groups.length ? new DataSet(groups) : undefined,
        options
      );

      const pick = (obj, k) => {
        if (obj[k] != null && String(obj[k]).trim() !== "") return { key: k, value: obj[k] };
        return null;
      };

      const maybeDate = (v) => {
        const t = Date.parse(v);
        return Number.isNaN(t)
          ? v
          : new Date(t).toLocaleDateString(undefined,
              { year: "numeric", month: "short", day: "numeric" });
      };

      const flat = (v) => (v && typeof v === "object") ? JSON.stringify(v) : v;

      function renderArticle(a) {
        if (!a || typeof a !== "object") return `<div class="vt-row"><b>${esc(a)}</b></div>`;
        const t  = pick(a, "Title");
        const dt = pick(a, "Date");
        const aut = pick(a, "Author")
        const pap = pick(a, "Newspaper")
        const skip = new Set([t && t.key, dt && dt.key, aut && aut.key, pap && pap.key].filter(Boolean));
        const body = Object.entries(a)
          .filter(([k]) => !skip.has(k))
          .map(([k, v]) =>
            `<div class="vt-sub"><div>${esc(flat(v))}</div></div>`)
          .join("");
        return `
          <details class="vt-article">
            <summary>
              <span class="vt-art-title">${esc(t ? t.value : "Untitled")}</span>
              ${dt ? `<span class="vt-art-date">${esc(maybeDate(dt.value))}</span>` : ""}
              <span class="vt-art-author">${esc(aut ? aut.value: "No author")}</span>
              <span class="vt-art-paper">${esc(pap ? pap.value: "No author")}</span>
            </summary>
            <div class="vt-art-body">${body}</div>
          </details>`;
      }

      function renderField(k, v) {
        if (Array.isArray(v)) {
          if (v.length === 0)
            return `<div class="vt-row"><span>${esc(k)}</span><b class="vt-none">none</b></div>`;
          if (v.every((x) => x === null || typeof x !== "object"))
            return `<div class="vt-row"><span>${esc(k)}</span><b>${esc(v.join(", "))}</b></div>`;
          return `
            <div class="vt-group">
              <div class="vt-group-head"><span class="vt-count">${v.length}</span> article(s) found.</div>
              ${v.map(renderArticle).join("")}
            </div>`;
        }
        if (v && typeof v === "object")
          return `
            <div class="vt-group">
              <div class="vt-group-head">${esc(k)}</div>
              ${Object.entries(v).map(([k2, v2]) => renderField(k2, v2)).join("")}
            </div>`;
        return `<div class="vt-row"><span>${esc(k)}</span><b>${esc(v)}</b></div>`;
      }
      
      function showDetail(id) {
        if (id == null) {
          panel.innerHTML = "<p class='vt-empty'>Select an event to see details.</p>";
          return;
        }
        const item = items.get(id) || {};
        const d = item.detail || {};

        const fmt = (s) => s
          ? new Date(s).toLocaleDateString(undefined,
              { year: "numeric", month: "short", day: "numeric" })
          : "";
        const when = item.end
          ? `${fmt(item.start)} \\u2013 ${fmt(item.end)}`
          : fmt(item.start);

        const rows = Object.entries(d).map(([k, v]) => renderField(k, v)).join("");

        panel.innerHTML = `
          <div class="vt-when">${esc(when)}</div>
          <p class="vt-body">${esc(item.content)}</p>
          ${rows}
        `;
      }

      showDetail(null);

      tl.on("select", (props) => {
        const id = props.items.length ? props.items[0] : null;
        showDetail(id);
        if (model.get("emit_selection")) {
          model.set("selected", props.items);
          model.save_changes();
        }
      });

      model.on("change:items", () => {
        items.clear();
        items.add(model.get("items"));
        showDetail(null);
      });

      return () => tl.destroy();
    }
    export default { render };
    """

    _css = """
    .vt-widget .vis-item .vis-item-content {
      white-space: normal;
      overflow-wrap: break-word;
      padding: 4px 6px;
      line-height: 1.3;
    }
    .vt-widget .vis-item.vis-box .vis-item-content,
    .vt-widget .vis-item.vis-point .vis-item-content { max-width: 240px; }
    .vt-widget .vis-item.vis-range { overflow: hidden; }
    .vt-widget .vis-item.vis-range .vis-item-content { max-width: 100%; }

    .vt-widget .vt-panel {
      margin-top: 12px;
      padding: 12px 14px;
      border: 1px solid rgba(128,128,128,0.35);
      border-radius: 6px;
      font-size: 0.9rem;
      line-height: 1.45;
      min-height: 4.5rem;
    }
    .vt-widget .vt-panel h3 { margin: 0 0 2px; font-size: 1rem; }
    .vt-widget .vt-when { opacity: 0.65; font-size: 0.8rem; margin-bottom: 8px; }
    .vt-widget .vt-row {
      display: flex; justify-content: space-between; gap: 1rem;
      padding: 3px 0; border-bottom: 1px solid rgba(128,128,128,0.15);
    }
    .vt-widget .vt-row span { opacity: 0.7; }
    .vt-widget .vt-empty { opacity: 0.55; margin: 0; }
    """

    items          = traitlets.List([]).tag(sync=True)
    groups         = traitlets.List([]).tag(sync=True)
    options        = traitlets.Dict({}).tag(sync=True)
    height         = traitlets.Unicode("500px").tag(sync=True)
    selected       = traitlets.List([]).tag(sync=True)
    emit_selection = traitlets.Bool(False).tag(sync=True)

In [50]:
[{k: v for k, v in i.items() if k in ("type", "start", "end")} for i in items[:5]]

[{'start': '1986-01-01T00:00:00',
  'type': 'range',
  'end': '1989-01-01T00:00:00'},
 {'start': '1986-01-01T00:00:00',
  'type': 'range',
  'end': '1989-01-01T00:00:00'},
 {'start': '1990-01-01T00:00:00',
  'type': 'range',
  'end': '1995-01-01T00:00:00'},
 {'start': '1987-11-01T00:00:00', 'type': 'box'},
 {'start': '1988-03-07T00:00:00', 'type': 'box'}]

In [51]:
tl = VisTimeline(items=items)
tl

In [21]:
s = pd.to_datetime([i["start"] for i in items], errors="coerce")
e = pd.to_datetime([i.get("end") for i in items], errors="coerce")
print(s.min(), s.max(), e.max())
print([i["id"] for i, ok in zip(items, s.notna()) if not ok])          # unparseable
print([i["id"] for i, d in zip(items, s) if pd.notna(d) and not (1900 < d.year < 2100)])

1986-01-01 00:00:00 2026-01-22 00:00:00 1995-01-01 00:00:00
[]
[]
